# HW5: Fixed-Income Valuation and First-Order Risk

Complete the five library TODOs, then run all cells.

In [1]:
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfCreateProductFixedAccruedCashflow,
    qfCreateValuationParameters,
    qfCreateValuationParametersCollection,
    qfCreateValueReport,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import IndexRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve import YieldCurveModelComponent
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductFixedAccrued

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells.
The funding component adds an IFR spread over SOFR.

In [2]:
VALUE_DATE = "2026-01-02"
PAYMENT_DATE = Date("2026-10-02")
FUNDING_INDEX = "SOFR-1B-FLAT"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-02", "2028-01-02"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2027-01-02"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_quotes.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Node date,
2026-04-02,0.030000
2027-01-02,0.040000
2028-01-02,0.050000


,SOFR-1B-FLAT IFR spread
Node date,
2026-07-02,0.000000
2027-01-02,0.001000


## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first and funding second.

In [3]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, funding_method])

## 3. Build the model

Copy IFR inputs into the model. The registered funding component references SOFR.

In [5]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
print("Component sizes:", model.num_sub_components)

Component sizes: [3, 2]


The following helpers retrieve a component (`retrieve_model_component`), allocate zero-filled
gradient blocks in component order (`resize_gradient`), and locate a component's block
(`retrieve_model_component_index`, using the `component_indices` map).

In [6]:
index = IndexRegistry().get("SOFR-1B")
component = cast(YieldCurveModelComponent, model.retrieve_model_component(index))
funding_component = model.retrieve_model_component(FUNDING_INDEX)

print("SOFR-1B ->", type(component).__name__, "|", component.component_identifier.name())
print("SOFR-1B-FLAT ->", funding_component.component_identifier.name())
display(pd.DataFrame(component.state_data.T, columns=["Node time (years)", "IFR"]))

SOFR-1B -> YieldCurveModelComponent | SOFRON Actual/360
SOFR-1B-FLAT -> SOFR-1B-FLAT


,Node time (years),IFR
0,0.246575,0.030000
1,1.000000,0.040000
2,1.999993,0.050000


In [7]:
gradient = []
model.resize_gradient(gradient)
print("[SOFR block, funding block]:", gradient)

[SOFR block, funding block]: [array([0., 0., 0.]), array([0., 0.])]


In [8]:
print("Stored map:", model.component_indices)
print("SOFR-1B -> block", model.retrieve_model_component_index("SOFR-1B"))
print("SOFR-1B-FLAT -> block", model.retrieve_model_component_index(FUNDING_INDEX))

Stored map: {'SOFRON Actual/360': 0, 'SOFR-1B-FLAT': 1}
SOFR-1B -> block 0
SOFR-1B-FLAT -> block 1


## 4. Discount-factor gradients

Complete **TODO 1** in `YieldCurveModelComponent` and **TODO 2** in `YieldCurve`
(`fixedincomelib/yield_curve/yield_curve_model.py`). Calculate the local derivative, then apply
the product rule recursively to $D_{\mathrm{funding}}=D_{\mathrm{spread}}D_{\mathrm{SOFR}}$.

Gradient methods write raw derivatives into the supplied container and return `None`.
`scaler` scales the new contribution; `accumulate=False` replaces the output and `True` adds to it.

With the default piecewise-constant interpolation, the third SOFR segment starts after
2027-01-02. Its gradient is zero for the 2026-10-02 payment.

In [9]:
local_gradient = np.zeros(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_gradient)
print("Local SOFR discount-factor gradient:", local_gradient)

Local SOFR discount-factor gradient: [-0.239898 -0.487793 -0.      ]


In [10]:
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, gradient)
print("Model gradient [SOFR block, funding block]:", gradient)

Model gradient [SOFR block, funding block]: [array([-0.239838, -0.48767 ,  0.      ]), array([-0.482341, -0.245168])]


## 5. Fixed-accrued valuation engine

Complete **TODOs 3–5** in `ValuationEngineProductFixedAccrued`: `__init__`, `calculate_value`,
and `calculate_risk` (`fixedincomelib/yield_curve/valuation_engine.py`).
Select the funding curve, then compute $C=N\alpha$, $PV=C D(T_p)$ and
$\nabla PV=C\nabla D(T_p)$. Notional is signed; this product has no separate fixed-rate field.

| Payment date | PV / cash today | Risk |
|---|---|---|
| Future | $C D(T_p)$ / 0 | $C\nabla D(T_p)$ |
| Today | $C$ / $C$ | 0 |
| Past | 0 / 0 | 0 |

In [11]:
product = qfCreateProductFixedAccruedCashflow(
    "2026-04-02", "2026-07-02", "USD", 1_000_000, "ACT/360",
    payment_date=PAYMENT_DATE.ISO(), business_day_convention="NONE",
)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Accrual fraction:", product.accrued)
print("Cash amount:", product.notional * product.accrued)

Accrual fraction: 0.25277777777777777
Cash amount: 252777.77777777778


In [12]:
engine = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
engine.calculate_value()

product_gradient = []
engine.calculate_risk(product_gradient)

print("PV:", engine.value)
print("Cash today:", engine.cash)
print("Product gradient [SOFR block, funding block]:", product_gradient)

PV: 245870.91057967892
Cash today: 0.0
Product gradient [SOFR block, funding block]: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


## 6. Reports

`qfCreateValueReport` calls the engine for `PV` or `FirstOrderRisk`.
`state_gradient` retains raw blocks; `display()` multiplies risk by `0.0001`
to show the signed PV change per positive 1 bp node shift.

In [13]:
print("PV report:", qfCreateValueReport(model, product, parameters, "PV"))
risk_report = qfCreateValueReport(model, product, parameters, "FirstOrderRisk")

print("Raw blocks:", risk_report.state_gradient)
display(risk_report.display()[["COMPONENT", "NODE_INDEX", "AXIS1", "UNIT", "VALUES"]])

PV report: [['USD', 245870.91057967892]]
Raw blocks: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


,COMPONENT,NODE_INDEX,AXIS1,UNIT,VALUES
0,SOFRON Actual/360,0,2026-04-02,0.000100,-6.062570
1,SOFRON Actual/360,1,2027-01-02,0.000100,-12.327226
2,SOFRON Actual/360,2,2028-01-02,0.000100,0.000000
3,SOFR-1B-FLAT,0,2026-07-02,0.000100,-12.192503
4,SOFR-1B-FLAT,1,2027-01-02,0.000100,-6.197294


## 7. Checks

Independent checks of the five completed TODOs. Section 7.1 compares every analytic gradient
(component, model and product) with central finite differences obtained by bumping each IFR node
through `perturb_model_parameter` and repricing. Section 7.2 verifies the product-rule decomposition
$\nabla D_{\mathrm{funding}} = \bigl(D_{\mathrm{spread}}\nabla D_{\mathrm{SOFR}},\; D_{\mathrm{SOFR}}\nabla D_{\mathrm{spread}}\bigr)$,
the `scaler`/`accumulate` semantics and the report's 1 bp scaling. Section 7.3 exercises the
same-day, past and short-notional cases of the engine.

In [14]:
# 7.1 Analytic gradients against central finite differences (bump and reprice).
from fixedincomelib import qfAccrued, qfDiscountFactor

bump = 1e-5
blocks = [("SOFR-1B", component), (FUNDING_INDEX, funding_component)]


def finite_difference(price, target, node):
    model.perturb_model_parameter(target, node, +bump)
    up = price()
    model.perturb_model_parameter(target, node, -2.0 * bump)
    down = price()
    model.perturb_model_parameter(target, node, +bump)  # restore the node
    return (up - down) / (2.0 * bump)


def fd_blocks(price):
    return [
        np.array([finite_difference(price, target, i) for i in range(comp.num_state_data)])
        for target, comp in blocks
    ]


# (a) Local SOFR gradient (TODO 1): a SOFR-only price has no funding-block risk.
fd_local = fd_blocks(lambda: component.discount_factor(PAYMENT_DATE))
assert np.allclose(fd_local[0], local_gradient, rtol=1e-6, atol=1e-10)
assert np.allclose(fd_local[1], 0.0)

# (b) Model gradient of the funding discount factor (TODO 2), both blocks.
fd_model = fd_blocks(lambda: qfDiscountFactor(model, FUNDING_INDEX, PAYMENT_DATE.ISO()))
for analytic, numeric in zip(gradient, fd_model):
    assert np.allclose(analytic, numeric, rtol=1e-6, atol=1e-10)


# (c) Product gradient (TODO 5): bump, re-run the engine, compare with calculate_risk.
def engine_pv():
    pricer = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
    pricer.calculate_value()
    return pricer.value


fd_product = fd_blocks(engine_pv)
for analytic, numeric in zip(product_gradient, fd_product):
    assert np.allclose(analytic, numeric, rtol=1e-6, atol=1e-6)
assert np.isclose(engine_pv(), engine.value)  # the model state is back where it started

check_table = pd.DataFrame({
    "Component": ["SOFR-1B"] * component.num_state_data
                 + [FUNDING_INDEX] * funding_component.num_state_data,
    "Node date": list(sofr_quotes.index) + list(funding_quotes.index),
    "Analytic dPV/df": np.concatenate(product_gradient),
    "FD dPV/df": np.concatenate(fd_product),
    "abs diff": np.abs(np.concatenate(product_gradient) - np.concatenate(fd_product)),
})
display(check_table)
print("Analytic gradients match finite differences.")

,Component,Node date,Analytic dPV/df,FD dPV/df,abs diff
0,SOFR-1B,2026-04-02,"-60,625.703979","-60,625.703979",0.000001
1,SOFR-1B,2027-01-02,"-123,272.264756","-123,272.264758",0.000001
2,SOFR-1B,2028-01-02,0.000000,0.000000,0.000000
3,SOFR-1B-FLAT,2026-07-02,"-121,925.026890","-121,925.026891",0.000001
4,SOFR-1B-FLAT,2027-01-02,"-61,972.941845","-61,972.941845",0.000001


Analytic gradients match finite differences.


In [15]:
# 7.2 Product rule, scaler/accumulate semantics and report scaling.
D_sofr = component.discount_factor(PAYMENT_DATE)
D_spread = funding_component.discount_factor(PAYMENT_DATE)
D_funding = qfDiscountFactor(model, FUNDING_INDEX, PAYMENT_DATE.ISO())
assert np.isclose(D_funding, D_sofr * D_spread)

# Each block is the *other* curve's factor times that component's local gradient ...
local_funding = np.zeros(funding_component.num_state_data)
funding_component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_funding)
assert np.allclose(gradient[0], D_spread * local_gradient)
assert np.allclose(gradient[1], D_sofr * local_funding)

# ... equivalently, -D_funding times the integral weights of each interpolator.
T = qfAccrued(VALUE_DATE, PAYMENT_DATE.ISO(), "ACT/ACT (ISDA)")
w_sofr = component.state_data_interpolator.gradient_of_integrated_value_wrt_ordinate(0.0, T)
w_funding = funding_component.state_data_interpolator.gradient_of_integrated_value_wrt_ordinate(0.0, T)
assert np.allclose(gradient[0], -D_funding * w_sofr)
assert np.allclose(gradient[1], -D_funding * w_funding)
assert w_sofr[2] == 0.0  # the third SOFR segment starts after the payment date

# Asking for SOFR-1B itself touches only the SOFR block.
sofr_only = []
model.discount_factor_gradient_wrt_state("SOFR-1B", PAYMENT_DATE, sofr_only)
assert np.allclose(sofr_only[0], local_gradient) and np.allclose(sofr_only[1], 0.0)

# scaler scales the new contribution; accumulate=True adds to the container, False replaces it.
scratch = []
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, scratch, scaler=2.0)
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, scratch, scaler=3.0, accumulate=True)
assert all(np.allclose(s, 5.0 * g) for s, g in zip(scratch, gradient))
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, scratch)  # overwrite
assert all(np.allclose(s, g) for s, g in zip(scratch, gradient))

# The same semantics hold at the component and product level.
scratch_local = np.ones(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, scratch_local, scaler=-1.0, accumulate=True)
assert np.allclose(scratch_local, 1.0 - local_gradient)
engine.calculate_risk(scratch, scaler=0.5, accumulate=True)
assert all(np.allclose(s, g + 0.5 * p) for s, g, p in zip(scratch, gradient, product_gradient))
assert all(np.allclose(p, engine.cash_amount * g) for p, g in zip(product_gradient, gradient))

# The report keeps raw blocks and shows 1 bp units in display().
assert all(np.allclose(r, p) for r, p in zip(risk_report.state_gradient, product_gradient))
assert np.allclose(risk_report.display()["VALUES"].to_numpy(), 1e-4 * np.concatenate(product_gradient))
print("Product rule, scaler/accumulate semantics and report scaling verified.")

Product rule, scaler/accumulate semantics and report scaling verified.


In [16]:
# 7.3 Same-day, past and short payments.
def value_and_risk(payment_date, notional=1_000_000):
    this_product = qfCreateProductFixedAccruedCashflow(
        "2026-04-02", "2026-07-02", "USD", notional, "ACT/360",
        payment_date=payment_date, business_day_convention="NONE",
    )
    this_engine = ValuationEngineProductFixedAccrued(model, parameters, this_product, ValuationRequest.PV)
    this_engine.calculate_value()
    this_gradient = []
    this_engine.calculate_risk(this_gradient)
    return this_engine, this_gradient


C = product.notional * product.accrued

# Payment today: PV = cash = C and no discount-factor risk.
today, g_today = value_and_risk(VALUE_DATE)
assert np.isclose(today.value, C) and np.isclose(today.cash, C)
assert all(np.all(block == 0.0) for block in g_today)

# Payment in the past: nothing left to value.
past, g_past = value_and_risk("2025-12-15")
assert past.value == 0.0 and past.cash == 0.0
assert all(np.all(block == 0.0) for block in g_past)

# A non-future payment leaves an accumulated gradient untouched.
kept = [block.copy() for block in product_gradient]
past.calculate_risk(kept, accumulate=True)
assert all(np.allclose(k, p) for k, p in zip(kept, product_gradient))

# Short position: the signed notional flips PV and risk.
short, g_short = value_and_risk(PAYMENT_DATE.ISO(), notional=-1_000_000)
assert np.isclose(short.value, -engine.value) and short.cash == 0.0
assert all(np.allclose(s, -p) for s, p in zip(g_short, product_gradient))

# The report entry point agrees with the engine.
pv_rows = qfCreateValueReport(model, product, parameters, "pv")
assert pv_rows[0][0] == "USD" and np.isclose(pv_rows[0][1], engine.value)

cases = pd.DataFrame({
    "Case": [f"Future ({PAYMENT_DATE.ISO()})", f"Today ({VALUE_DATE})", "Past (2025-12-15)",
             f"Short, future ({PAYMENT_DATE.ISO()})"],
    "PV": [engine.value, today.value, past.value, short.value],
    "Cash today": [engine.cash, today.cash, past.cash, short.cash],
    "Sum of |risk|": [sum(np.abs(b).sum() for b in g)
                      for g in (product_gradient, g_today, g_past, g_short)],
})
display(cases)
print("Same-day, past and short cases behave as specified.")

,Case,PV,Cash today,Sum of |risk|
0,Future (2026-10-02),"245,870.910580",0.000000,"367,795.937470"
1,Today (2026-01-02),"252,777.777778","252,777.777778",0.000000
2,Past (2025-12-15),0.000000,0.000000,0.000000
3,"Short, future (2026-10-02)","-245,870.910580",0.000000,"367,795.937470"


Same-day, past and short cases behave as specified.
